In [2]:
%pip install anthropic python-dotenv

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 1.2 MB 1.5 MB/s eta 0:00:01
     |████████████████████████████████| 73 kB 6.8 MB/s  eta 0:00:01
     |████████████████████████████████| 472 kB 11.0 MB/s eta 0:00:01
     |████████████████████████████████| 113 kB 10.8 MB/s eta 0:00:01
     |████████████████████████████████| 315 kB 10.2 MB/s eta 0:00:01
     |████████████████████████████████| 69 kB 9.7 MB/s  eta 0:00:01
     |████████████████████████████████| 78 kB 10.7 MB/s eta 0:00:01
     |████████████████████████████████| 136 kB 10.9 MB/s eta 0:00:01
     |████████████████████████████████| 1.9 MB 1.1 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [7]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-5-5"

In [8]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return "".join(block.text for block in message.content if block.type == "text")

In [9]:
# Function to generate a new dataset
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex",
        "solution_criteria": "Key criteria for evaluating the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    text = chat(
        messages,
        system="Return only a valid JSON array. Do not include Markdown code fences or commentary.",
    )
    return json.loads(text)

In [10]:
# Generate the dataset and write it to 'dataset.json'
dataset = generate_dataset()
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [11]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    eval_text = chat(
        messages,
        system="Return only a valid JSON object. Do not include Markdown code fences or commentary.",
    )
    return json.loads(eval_text)

In [12]:
# Passes a test case into Claude
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""

    messages = []
    add_user_message(messages, prompt)
    output = chat(
        messages,
        system="Return only the requested Python, JSON, or regex. Do not include Markdown code fences or commentary.",
    )
    return output

In [13]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


In [14]:
# Function to execute a single test case and grade the output
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [15]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [16]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 7.5


In [17]:
print(json.dumps(results, indent=2))

[
  {
    "output": "```python\nimport boto3\nfrom botocore.exceptions import ClientError\n\ndef has_default_encryption(bucket_name):\n    s3 = boto3.client(\"s3\")\n    try:\n        response = s3.get_bucket_encryption(Bucket=bucket_name)\n        rules = response.get(\"ServerSideEncryptionConfiguration\", {}).get(\"Rules\", [])\n        return any(\n            rule.get(\"ApplyServerSideEncryptionByDefault\", {}).get(\"SSEAlgorithm\")\n            for rule in rules\n        )\n    except ClientError as e:\n        if e.response[\"Error\"][\"Code\"] == \"ServerSideEncryptionConfigurationNotFoundError\":\n            return False\n        raise\n```",
    "test_case": {
      "task": "Write a Python function that takes an S3 bucket name and returns True if the bucket has server-side encryption enabled by default, using boto3. Handle the case where no encryption configuration exists by returning False.",
      "format": "python",
      "solution_criteria": "Uses boto3 s3 client get_buck